# 교차 검증 (Cross Validation)
데이터를 여러 번 나눠서 학습/평가를 반복하는 방법이다.

한 번만 분할해서 평가하면 우연히 분할이 잘되거나 잘못되어 모델 성능이 왜곡될 수 있다.

교차 검증을 사용하면 더 안정적인 평가가 가능하다.

## K-Fold 교차 검증
데이터를 K등분한 후 각각 한 번씩 테스트 데이터로 사용한다.

1. 데이터를 K개의 폴드로 나눈다.
2. K번 학습/평가를 반복한다.
3. 매번 다른 폴드를 테스트 데이터로 사용한다.
4. K번의 평가 점수를 평균낸다.

보통 K=5 또는 K=10을 사용한다.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_score

data = load_iris()
X, y = data.data, data.target

model = LogisticRegression(max_iter=200)

# K-Fold 5회
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kf)

print(f"각 폴드 점수: {scores}")
print(f"평균 점수: {scores.mean():.4f}")
print(f"표준편차: {scores.std():.4f}")

## Stratified K-Fold
각 폴드에 클래스 비율을 유지하면서 분할한다.

분류 문제에서 권장된다.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=skf)

print(f"평균 점수: {scores.mean():.4f}")

## Leave-One-Out (LOO)
데이터 1개씩만 테스트로 사용하고 나머지는 학습으로 사용한다.

데이터 수가 N이라면 N번 학습/평가를 반복한다.

**데이터가 매우 적을 때만 사용** (시간이 오래 걸림)

In [ ]:
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()
scores = cross_val_score(model, X, y, cv=loo)

print(f"평균 점수: {scores.mean():.4f}")

## TimeSeriesSplit
시계열 데이터에 사용한다. 과거 데이터로 학습, 미래 데이터로 평가하도록 분할한다.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(model, X, y, cv=tscv)

## GridSearchCV로 하이퍼파라미터 튜닝
교차 검증을 활용하여 최적의 하이퍼파라미터를 찾는다.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10, None]
}

model = RandomForestClassifier(random_state=42)
grid = GridSearchCV(model, params, cv=5)
grid.fit(X, y)

print(f"최적 파라미터: {grid.best_params_}")
print(f"최고 점수: {grid.best_score_:.4f}")

## 어떤 교차 검증을 써야 할까?

| 상황 | 권장 |
| --- | --- |
| 일반 분류/회귀 | K-Fold (K=5) |
| 분류 (불균형 데이터 포함) | Stratified K-Fold |
| 데이터가 매우 적음 | Leave-One-Out |
| 시계열 데이터 | TimeSeriesSplit |